In [3]:
""" Project 1: Aspect Based Sentiment Analysis from Text 

E-commerce site:

Sentiment: Positive, Neutral, Negative
Review: The price was too high. But I loved the product.

Task: Sentiment Analysis from Text
Sentiment: Positive.

Task: Aspect Based Sentiment Analysis from Text
Price: Negative
Product: Positive

Motivation: E-commerce 
"""

train_path = r"C:\Users\Loccha kakko\PyCharmMiscProject\machine_learning\data\train.csv"
test_path = r"C:\Users\Loccha kakko\PyCharmMiscProject\machine_learning\data\test.csv"

In [4]:
# Read the dataset
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import pandas as pd
import os
import torch.nn as nn
import pytorch_lightning as pl
from mlflow.models import infer_signature
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_df.head()




,review,aspect,sentiment
0,But the staff was so horrible to us .,staff,negative
1,"To be completely fair , the only redeeming fac...",food,positive
2,"The food is uniformly exceptional , with a ver...",food,positive
3,"The food is uniformly exceptional , with a ver...",kitchen,positive
4,"The food is uniformly exceptional , with a ver...",menu,neutral


In [5]:
ARTIFACT_FOLDER_NAME = "my_model" # Directory to save models
SOURCE_CODE_PATH = os.path.join(
        os.getcwd(),
        "project_1_absa.ipynb",
) # Our current notebook file path

SOURCE_CODE_ARTIFACT = "trainer.ipynb" # Filename to save our source code

In [6]:
train_df.value_counts(['sentiment'])

sentiment
positive     2164
negative      807
neutral       637
Name: count, dtype: int64

In [7]:
train_df.value_counts(['aspect'])

aspect                                          
food                                                340
service                                             189
place                                                59
menu                                                 56
prices                                               56
                                                   ... 
white tuna sashimi                                    1
whitefish                                             1
whitefish salad                                       1
whole grilled fish                                    1
wild mushroom ( third generation-Fornini ) pizza      1
Name: count, Length: 1322, dtype: int64

In [8]:
"""
Data
Model
Train
"""

index = 0
example = train_df.iloc[index]

print("Review:", example['review'])
print("Aspect:", example['aspect'])
print("Sentiment:", example['sentiment'])

Review: But the staff was so horrible to us .
Aspect: staff
Sentiment: negative


In [9]:
import re

# Normalize the text

text = example['review']

#convert the text into lowercase
text = text.lower()

# remove punctuations, special characters
text = re.sub(r'[^a-z0-9\s]', '', text)

print(text.split())
# remove extra whitespaces
text = ' '.join(text.split())

text

['but', 'the', 'staff', 'was', 'so', 'horrible', 'to', 'us']


'but the staff was so horrible to us'

In [10]:
# Normalize the text

def normalize(text):    
    #convert the text into lowercase
    text = text.lower()    
    # remove punctuations, special characters
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # remove extra whitespaces
    text = ' '.join(text.split())
    return text
    
text = example['review']

normalized_text = normalize(text)
print(normalized_text)

but the staff was so horrible to us


In [11]:
# Tokenization

# Token is any sequential part of a text that collectively make up the entire text
# For example: text = but the staff was so horrible to us
# Tokens: "but", "the", "staff", "was", "so", "horrible", "to", "use" (word level tokenization)
# Standard tokenization technique: Character-level tokenization
# For example: text = "Life is good"
# Character level tokens: 'L', 'i', 'f', 'e', ' ', 'g', 'o', 'o', 'd'
# Advanced tokenization technique: BPE (Byte-Pair Encoding)
# For example: text = "life is good"
# Bypte pair tokens: li, fe, 'is', 'goo', 'd'

def tokenize(text):
    # Word level tokenization
    tokens = text.split()
    return tokens

tokens = tokenize(normalized_text)
print(tokens)

['but', 'the', 'staff', 'was', 'so', 'horrible', 'to', 'us']


In [12]:
def build_vocab(texts):
    token_2_id = {
        '<PAD>': 0,
        '<UNK>': 1,
    }
    
    idx = 2
    for text in texts:
        normalized_text = normalize(text)
        tokens = tokenize(normalized_text)
        
        for token in tokens:
            if token_2_id.get(token) is None:
                token_2_id[token] = idx
                idx += 1
    return token_2_id

token_2_id = build_vocab(train_df['review'])
print("Vocabulary size:", len(token_2_id))

Vocabulary size: 3800


In [13]:
def convert_tokens_2_ids(tokens):
    input_ids = [
        token_2_id.get(token, token_2_id['<UNK>']) for token in tokens
    ]
    return input_ids
text = example['review'] + " hello "
normalized_text = normalize(text)
tokens = tokenize(normalized_text)
input_ids = convert_tokens_2_ids(tokens)

print("Vocabulary size:", len(token_2_id))
print(tokens)
print(input_ids)

Vocabulary size: 3800
['but', 'the', 'staff', 'was', 'so', 'horrible', 'to', 'us', 'hello']
[2, 3, 4, 5, 6, 7, 8, 9, 1]


In [14]:
input_ids = [token_2_id.get(token, token_2_id['<UNK>']) 
             for token in tokens]
print(input_ids)

[2, 3, 4, 5, 6, 7, 8, 9, 1]


In [15]:
import pickle

# Save the dictionary
with open("vocab.pkl", "wb") as f:
    pickle.dump(token_2_id, f)

In [16]:
# Label mapping

label_map = {
    'negative': 0,
    'neutral': 1,
    'positive': 2,
}

sentiment = 'positive'
label_id = label_map[sentiment]

print("Label id:", label_id)

Label id: 2


In [17]:
# input: f(review, aspect) => sentiment (label)

text_aspect_pair = example['review'] + ' ' + example['aspect']
text_aspect_pair

'But the staff was so horrible to us . staff'

In [18]:
def __getitem__(idx):
    example = train_df.iloc[idx]
    text = example['review']
    aspect = example['aspect']
    sentiment = example['sentiment']
    
    text_aspect_pair = text + ' ' + aspect
    normalized_text = normalize(text_aspect_pair)
    tokens = tokenize(normalized_text)
    input_ids = convert_tokens_2_ids(tokens)
    label_id = label_map[sentiment]
    return {
        "input_ids": input_ids, 
        "label": label_id
    }

processed_example = __getitem__(idx=0)

processed_example

{'input_ids': [2, 3, 4, 5, 6, 7, 8, 9, 4], 'label': 0}

#### Data

In [19]:
from torch.utils.data import Dataset
import torch

class ABSADataset(Dataset):
    def __init__(self, df):
        self.df = df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        example = train_df.iloc[idx]
        text = example['review']
        aspect = example['aspect']
        sentiment = example['sentiment']
        
        text_aspect_pair = text + ' ' + aspect
        normalized_text = normalize(text_aspect_pair)
        tokens = tokenize(normalized_text)
        input_ids = convert_tokens_2_ids(tokens)
        label_id = label_map[sentiment]
        return {
            "input_ids": input_ids, 
            "label": label_id
        }
    
    @staticmethod
    def collate_fn(batch):
        batch_input_ids = [item['input_ids'] for item in batch]
        batch_labels = [item['label'] for item in batch]
        
        max_len = max(
            len(input_ids) for input_ids in batch_input_ids)
        
        pad_token_id = token_2_id['<PAD>'] 
        batch_padded_input_ids = [
            input_ids + [pad_token_id] * (max_len - len(input_ids)) for input_ids in batch_input_ids
        ]
        
        return {
            "batch_input_ids": torch.tensor(
                batch_padded_input_ids, dtype=torch.long),
            "batch_labels": torch.tensor(batch_labels, dtype=torch.long)
        }
    
# Batch: Is a collection of examples
# Typically 32, 64, 128 etc. 

# [125, 124, 24]
# [147, 24, 369, 789]
# If batch size is 2, then max_len = 4
# We padd input_ids 1 with <PAD> token
# [147, 124, 24, 0]

In [20]:
train_ds = ABSADataset(train_df)
train_ds.__getitem__(0)

{'input_ids': [2, 3, 4, 5, 6, 7, 8, 9, 4], 'label': 0}

In [21]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader

class ABSADataModule(pl.LightningDataModule):
    def __init__(self, train_path, test_path, batch_size):
        super().__init__()
        self.train_path = train_path
        self.test_path = test_path
        self.batch_size = batch_size
    
    def setup(self, stage=None):
        # prepare the dataset for training
        # You may download, process some stuff
        # Everything data related we need to do before training
        # Read the dataset
        train_df = pd.read_csv(self.train_path)
        test_df = pd.read_csv(self.test_path)
        
        # build vocabulary
        self.token_2_id = build_vocab(train_df['review'])
        
        # Initialize the dataset
        self.train_set = ABSADataset(train_df)
        self.test_set = ABSADataset(test_df)
    
    def train_dataloader(self):
        return DataLoader(
            self.train_set,
            batch_size=self.batch_size,
            shuffle=True,
            collate_fn=ABSADataset.collate_fn
        )
    def val_dataloader(self):
        return DataLoader(
            self.test_set,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=ABSADataset.collate_fn
        )
    
    def test_dataloader(self):
        return DataLoader(
            self.test_set,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=ABSADataset.collate_fn
        )

In [22]:
data_module = ABSADataModule(
    train_path = r"C:\Users\Loccha kakko\PyCharmMiscProject\machine_learning\data\train.csv",
    test_path = r"C:\Users\Loccha kakko\PyCharmMiscProject\machine_learning\data\test.csv",
    batch_size=32
)

data_module.setup()

### Model

In [35]:
from torch import optim
from torchmetrics.classification import Accuracy
import torch.nn as nn

class ABSAModel(pl.LightningModule):
    def __init__(self, vocab_size, num_labels=3):
        super().__init__()

        self.vocab_size = vocab_size
        self.num_labels = num_labels

        # Embedding: Convert each token id is represented  by a vector
        # For example: input_ids = [124, 14, 35]
        # input ids shape = (B, 3,)
        # After that, shape = (B, 3, 256)
        self.embedding_layer = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=256
        )

        # Sequence to sequence learning
        self.lstm_layer = nn.LSTM(
            input_size=256,
            hidden_size=512,
            batch_first=True
        )

        self.fc_layer = nn.Linear(
            in_features=512,
            out_features=num_labels
        )

        self.loss_fn = nn.CrossEntropyLoss()
        self.save_hyperparameters()

    def forward(self, x):
        embeddings = self.embedding_layer(x)
        lstm_out, _ = self.lstm_layer(embeddings)
        logits = self.fc_layer(lstm_out[:, -1, :])
        return logits

    def training_step(self, batch, batch_idx):
        input_ids = batch['batch_input_ids']
        labels = batch['batch_labels']
        logits = self(input_ids) # Calls the forward method
        loss = self.loss_fn(logits, labels)
        self.log('train_loss', loss, prog_bar=True)
        acc = self.compute_metrics(logits, labels)
        self.log('train_acc', acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids = batch['batch_input_ids']
        labels = batch['batch_labels']

        logits = self(input_ids) # Calls the forward method
        loss = self.loss_fn(logits, labels)
        self.log('val_loss', loss, prog_bar=True)
        acc = self.compute_metrics(logits, labels)
        self.log('val_acc', acc, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        input_ids = batch['batch_input_ids']
        labels = batch['batch_labels']

        logits = self(input_ids) # Calls the forward method
        loss = self.loss_fn(logits, labels)
        self.log('test_loss', loss, prog_bar=True)
        acc = self.compute_metrics(logits, labels)
        self.log('test_acc', acc, prog_bar=True)
        return loss

    def configure_optimizers(self) :
        return optim.Adam(self.parameters(), lr=3e-4)

    def compute_metrics(self, logits, labels):
        preds = logits.argmax(dim=1)
        accuracy = Accuracy(task="multiclass", num_classes=self.num_labels)
        return accuracy(preds, labels)

In [36]:
model = ABSAModel(
    vocab_size=len(data_module.token_2_id),
    num_labels=3
)

In [37]:
early_stopping = EarlyStopping(
    monitor='val_loss', # Should match with the validation step log key
    patience=2,
    verbose=True,
)

In [38]:
checkpoint_callback = ModelCheckpoint(
    monitor='val_acc', # Should match with the validation step log key
    save_top_k=1, # Saves top one model
    mode='max', # top means max validation accuracy
)

checkpoint_path = os.path.join(
    os.getcwd(), "checkpoints", "best_model.pth"
)

In [40]:
print(checkpoint_path)

C:\Users\Loccha kakko\PyCharmMiscProject\machine_learning\checkpoints\best_model.pth


### Training

In [41]:
trainer = pl.Trainer(
    max_epochs=30,
    callbacks=[checkpoint_callback, early_stopping], # Runs these checkpoints after each epochs by default
)

trainer.fit(model, data_module)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | embedding_layer | Embedding        | 972 K  | train
1 | lstm_layer      | LSTM             | 1.6 M  | train
2 | fc_layer        | Linear           | 1.5 K  | train
3 | loss_fn         | CrossEntropyLoss | 0      | train
-------------------------------------------------------------
2.6 M     Trainable params
0         Non-trainable params
2.6 M     Total params
10.205    Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\Loccha kakko\PyCharmMiscProject\machine_learning\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


C:\Users\Loccha kakko\PyCharmMiscProject\machine_learning\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 113/113 [00:29<00:00,  3.85it/s, v_num=42, train_loss=0.731, train_acc=0.792]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 0: 100%|██████████| 113/113 [00:32<00:00,  3.50it/s, v_num=42, train_loss=0.731, train_acc=0.792, val_loss=0.897, val_acc=0.643]

...

Epoch 21: 100%|██████████| 113/113 [00:18<00:00,  6.07it/s, v_num=42, train_loss=0.0511, train_acc=1.000, val_loss=0.0474, val_acc=0.988] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 21: 100%|██████████| 113/113 [00:20<00:00,  5.55it/s, v_num=42, train_loss=0.0511, train_acc=1.000, val_loss=0.0408, val_acc=0.987]

Metric val_loss improved by 0.007 >= min_delta = 0.0. New best score: 0.041


Epoch 22: 100%|██████████| 113/113 [00:18<00:00,  6.28it/s, v_num=42, train_loss=0.0207, train_acc=1.000, val_loss=0.0408, val_acc=0.987] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 23: 100%|██████████| 113/113 [00:18<00:00,  6.22it/s, v_num=42, train_loss=0.00904, train_acc=1.000, val_loss=0.0421, val_acc=0.988]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 23: 100%|██████████| 113/113 [00:19<00:00,  5.66it/s, v_num=42, train_loss=0.00904, train_acc=1.000, val_loss=0.077, val_acc=0.976] 

Monitored metric val_loss did not improve in the last 2 records. Best score: 0.041. Signaling Trainer to stop.


Epoch 23: 100%|██████████| 113/113 [00:19<00:00,  5.65it/s, v_num=42, train_loss=0.00904, train_acc=1.000, val_loss=0.077, val_acc=0.976]


### Evaluate

In [42]:
trainer.test(model, data_module)

C:\Users\Loccha kakko\PyCharmMiscProject\machine_learning\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 35/35 [00:02<00:00, 17.41it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9758713245391846     │
│         test_loss         │    0.07703319191932678    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.07703319191932678, 'test_acc': 0.9758713245391846}]

In [43]:
torch.save(model.state_dict(), "model_weights.pth")


In [44]:
save_path = os.path.abspath("model_weights.pth")
torch.save(model.state_dict(), save_path)

print("Model saved at:", save_path)

Model saved at: C:\Users\Loccha kakko\PyCharmMiscProject\machine_learning\model_weights.pth


In [139]:
"""
MLflow is a popular open source framework
for ML application management, experiment tracking, easy deployement.
"""

import mlflow
import mlflow.pytorch
from mlflow.models.signature import infer_signature

In [140]:
mlflow.set_experiment(experiment_name="ABSA")

<Experiment: artifact_location='file:///C:/Users/Loccha%20kakko/PyCharmMiscProject/machine_learning/mlruns/280134358175700971', creation_time=1764221443481, experiment_id='280134358175700971', last_update_time=1764221443481, lifecycle_stage='active', name='ABSA', tags={}>

In [142]:


with mlflow.start_run():
    # Log Hyperparameters
    mlflow.log_param("learning_rate", model.hparams.learning_rate)
    mlflow.log_param("batch_size", data_module.batch_size)
    mlflow.log_param("epochs", trainer.max_epochs)
    mlflow.log_param("vocab_size", model.vocab_size)
    mlflow.log_param("Labels", model.num_labels)


    # Train the model
    trainer.fit(model, data_module)

    # Get the best model
    best_model_path = checkpoint_callback.best_model_path
    best_model = ABSAModel.load_from_checkpoint(best_model_path)

    # Evaluate the model on the test set
    evaluation_score = trainer.test(model, data_module)

    mlflow.log_metric("test_accuracy", evaluation_score[0]["test_acc"])
    mlflow.log_metric("test_loss", evaluation_score[0]["test_loss"])

    # Save the model
    batch = next(iter(data_module.test_dataloader()))
    input_ids_example = batch["batch_input_ids"].cpu().numpy()

    # Forward pass to get example output
    pred_example = model(batch["batch_input_ids"].to(model.device)).detach().cpu().numpy()

    # Create signature
    signature = infer_signature(input_ids_example, pred_example)
    mlflow.pytorch.log_model(
    pytorch_model=model,
    artifact_path="ARTIFACT_FOLDER_NAME",
    input_example=input_ids_example,
    signature=signature
    )

    # Log the source code
    import shutil
    shutil.copyfile(SOURCE_CODE_PATH, SOURCE_CODE_ARTIFACT)
    mlflow.log_artifact(SOURCE_CODE_ARTIFACT)

AttributeError: 'ABSAModel' object has no attribute 'learning_rate'

In [160]:
print(f"mlflow ui --backend-store-uri {mlflow.get_tracking_uri()}")

mlflow ui --backend-store-uri file:///C:/Users/Loccha%20kakko/PyCharmMiscProject/machine_learning/mlruns
